In [ ]:
# !pip install pyproj
import pandas as pd
from pyproj import Transformer
import os

# ==========================================
# 1. Configuration and Constants
# ==========================================

# Base Paths (Uncomment the active path)
BASE_PATH = 'E:\\Projetos\\ABM-WP' # Currently active path

# Coordinate Reference Systems (CRS)
# EPSG:4326  = WGS84 (Latitude/Longitude)
# EPSG:31984 = SIRGAS 2000 / UTM zone 24S (Projected X/Y in meters)
CRS_SOURCE = "EPSG:4326"
CRS_TARGET = "EPSG:31984"

def main():
    print("--- Starting Coordinate Conversion ---")

    # ==========================================
    # 2. Load Data
    # ==========================================
    input_csv_path = os.path.join(BASE_PATH, 'includes', 'Tabela_consumidores_Itapua.csv')
    output_csv_path = os.path.join(BASE_PATH, 'includes', 'Tabela_consumidores_Itapua_convertida.csv')

    print(f"Reading file: {input_csv_path}")
    df = pd.read_csv(input_csv_path, delimiter=";")

    # ==========================================
    # 3. Coordinate Transformation
    # ==========================================
    print(f"Transforming coordinates from {CRS_SOURCE} to {CRS_TARGET}...")

    # Initialize Transformer
    # always_xy=True ensures the order is (Longitude, Latitude) -> (X, Y)
    transformer = Transformer.from_crs(CRS_SOURCE, CRS_TARGET, always_xy=True)

    # Vectorized transformation (much faster than row-by-row apply)
    x_values, y_values = transformer.transform(
        df["LONG_GEO"].values, 
        df["LAT_GEO"].values
    )

    # Assign new columns
    df["X"] = x_values
    df["Y"] = y_values

    # ==========================================
    # 4. Cleaning and Saving
    # ==========================================
    
    # Drop rows where transformation failed (NaN results)
    original_count = len(df)
    df = df.dropna(subset=["X", "Y"])
    
    # Save to CSV
    df.to_csv(output_csv_path, sep=";", index=False)

    print("-" * 30)
    print(f"Processing Complete.")
    print(f"Original records: {original_count}")
    print(f"Saved records: {len(df)}")
    print(f"File saved to: {output_csv_path}")

if __name__ == "__main__":
    main()

--- Starting Coordinate Conversion ---
Reading file: E:\Projetos\ABMS-WP\includes\Tabela_consumidores_Itapua.csv
Transforming coordinates from EPSG:4326 to EPSG:31984...
------------------------------
Processing Complete.
Original records: 16193
Saved records: 15332
File saved to: E:\Projetos\ABMS-WP\includes\Tabela_consumidores_Itapua_convertida.csv
